In [ ]:
import torch
print(f"CUDA 是否可用: {torch.cuda.is_available()}")
# 输出应当为 True

In [4]:
!pip install "git+https://github.com.cnpmjs.org/facebookresearch/pytorch3d.git"

Looking in indexes: https://mirrors.cloud.aliyuncs.com/pypi/simple, https://pypi.org/simple
  Cloning https://github.com.cnpmjs.org/facebookresearch/pytorch3d.git to /tmp/pip-req-build-5_yqo2ew
  Running command git clone --filter=blob:none --quiet https://github.com.cnpmjs.org/facebookresearch/pytorch3d.git /tmp/pip-req-build-5_yqo2ew
  fatal: 无法访问 'https://github.com.cnpmjs.org/facebookresearch/pytorch3d.git/'：Could not resolve host: github.com.cnpmjs.org
  error: subprocess-exited-with-error
  
  × git clone --filter=blob:none --quiet https://github.com.cnpmjs.org/facebookresearch/pytorch3d.git /tmp/pip-req-build-5_yqo2ew did not run successfully.
  │ exit code: 128
  ╰─> See above for output.
  
  note: This error originates from a subprocess, and is likely not a problem with pip.
error: subprocess-exited-with-error

× git clone --filter=blob:none --quiet https://github.com.cnpmjs.org/facebookresearch/pytorch3d.git /tmp/pip-req-build-5_yqo2ew did not run successfully.
│ exit code: 

In [2]:
# 1. 升级包管理工具
!pip install --upgrade pip

# 2. 安装前置依赖项与加速器（ninja 用于多核加速编译）
!pip install fvcore iopath matplotlib ninja

# 3. 使用 Gitee 链接直接源码编译安装
# ⚠️ 注意：加入 --no-build-isolation 可以防止某些云平台的严格沙箱隔离导致的编译报错
# 整个过程大约需要 5-10 分钟，请耐心等待！
!pip install "git+https://gitee.com/hongwenzhang/pytorch3d.git" --no-build-isolation

Looking in indexes: https://mirrors.cloud.aliyuncs.com/pypi/simple
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 42.8 MB/s eta 0:00:00a 0:00:01
  Attempting uninstall: pip
    Found existing installation: pip 23.3.2
    Uninstalling pip-23.3.2:
      Successfully uninstalled pip-23.3.2
Looking in indexes: https://mirrors.cloud.aliyuncs.com/pypi/simple
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for fvcore: filename=fvcore-0.1.5.post20221221-py3-none-any.whl size=61443 sha256=6680ee921a14eaeb2e199bb8c7093dd8c75bfa6b1317b5041e28cfe0a752687e
  Stored in directory: /root/.cache/pip/wheels/3a/e3/70/bd8fda7b2cf8b67e234290f8a72c8124b7e8e38c87451d1bbe
  Created wheel for iopath: filename=iopath-0.1.10-py3-none-any.whl size=31596 sha256=86e500bb0

In [5]:
import pytorch3d
import torch
print("PyTorch3D 版本:", pytorch3d.__version__)
print("CUDA可用:", torch.cuda.is_available())
print("GPU设备:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

PyTorch3D 版本: 0.7.9
CUDA可用: True
GPU设备: NVIDIA A10


In [6]:
import os
print(os.listdir("./"))

['.configs', '.ipynb_checkpoints', '.qwen', '.virtual_documents', 'cow.obj', 'output_meshes', 'work6.ipynb']


In [ ]:
%matplotlib inline
import os
import torch
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import clear_output 

import pytorch3d
# 引入 save_obj 用于保存 3D 模型
from pytorch3d.io import load_obj, save_obj
from pytorch3d.structures import Meshes
from pytorch3d.utils import ico_sphere
from pytorch3d.loss import mesh_edge_loss, mesh_laplacian_smoothing, mesh_normal_consistency
from pytorch3d.renderer import (
    look_at_view_transform, FoVPerspectiveCameras,
    RasterizationSettings, MeshRasterizer, SoftSilhouetteShader, BlendParams
)

# 确认设备
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(f"当前运行设备: {device}")
print(f"PyTorch3D 版本: {pytorch3d.__version__}")

# ---------------------------------------------------------
# 1. 直接读取助教打包好的本地模型文件
# ---------------------------------------------------------
obj_path = "cow.obj" 
if not os.path.exists(obj_path):
    raise FileNotFoundError("未找到 cow.obj，请确保代码文件与 obj 文件在同一目录下！")

# 准备目标数据与归一化处理
verts, faces, _ = load_obj(obj_path)
faces_idx = faces.verts_idx.to(device)
verts = verts.to(device)
verts = (verts - verts.mean(0)) / max(verts.abs().max(0)[0]) 
cow_mesh = Meshes(verts=[verts], faces=[faces_idx])

# ---------------------------------------------------------
# 2. 渲染管线与摄像机配置
# ---------------------------------------------------------
num_views = 20
cameras = FoVPerspectiveCameras(device=device, 
                                R=look_at_view_transform(2.7, torch.zeros(num_views), torch.linspace(-180, 180, num_views))[0], 
                                T=look_at_view_transform(2.7, torch.zeros(num_views), torch.linspace(-180, 180, num_views))[1])

rasterizer = MeshRasterizer(cameras=cameras, 
                            raster_settings=RasterizationSettings(image_size=256, blur_radius=np.log(1./1e-4 - 1.)*1e-4, faces_per_pixel=50))
shader = SoftSilhouetteShader(blend_params=BlendParams(sigma=1e-4, gamma=1e-4))

target_silhouette = shader(rasterizer(cow_mesh.extend(num_views)), cow_mesh.extend(num_views))[..., 3]

# ---------------------------------------------------------
# 3. 优化器初始化：从圆球开始
# ---------------------------------------------------------
src_mesh = ico_sphere(4, device)
deform_verts = torch.zeros_like(src_mesh.verts_packed(), requires_grad=True)
optimizer = torch.optim.SGD([deform_verts], lr=1.0, momentum=0.9)

# 创建保存中间结果的文件夹
output_dir = "output_meshes"
os.makedirs(output_dir, exist_ok=True)
# 新建图片存储文件夹
img_dir = "train_silhouette_imgs"
os.makedirs(img_dir, exist_ok=True)
print(f"中间模型将保存在目录: ./{output_dir}/")
print(f"剪影对比图将保存在目录: ./{img_dir}/")

# ---------------------------------------------------------
# 4. 可微渲染优化循环
# ---------------------------------------------------------
epochs = 300
for i in range(epochs):
    optimizer.zero_grad()
    
    # 依据当前计算出的偏移量，形变生成新的 Mesh
    new_src_mesh = src_mesh.offset_verts(deform_verts)
    
    # 渲染当前 Mesh 的 20 个视角剪影
    pred_silhouette = shader(rasterizer(new_src_mesh.extend(num_views)), new_src_mesh.extend(num_views))[..., 3]
    
    # 【核心损失】：均方误差 + 三大正则化惩罚
    loss_silhouette = ((pred_silhouette - target_silhouette) ** 2).mean()
    loss = loss_silhouette + \
           1.0 * mesh_laplacian_smoothing(new_src_mesh) + \
           0.1 * mesh_edge_loss(new_src_mesh) + \
           0.01 * mesh_normal_consistency(new_src_mesh)
    
    loss.backward()
    optimizer.step()

    # 可视化与模型保存逻辑：每 20 步执行一次
    if i % 20 == 0 or i == epochs - 1:
        clear_output(wait=True)
        print(f"迭代步数: {i:03d}/{epochs} | 总 Loss: {loss.item():.4f} | 剪影误差: {loss_silhouette.item():.4f}")
        
        # 提取当前网格的顶点和面，并保存为 .obj 文件
        current_verts = new_src_mesh.verts_list()[0]
        current_faces = new_src_mesh.faces_list()[0]
        
        # 修改文件名，最终输出 mesh_epoch_300.obj
        save_path = os.path.join(output_dir, f"mesh_epoch_{i+1:03d}.obj")
        save_obj(save_path, current_verts, current_faces)
        print(f"[*] 已保存当前 3D 模型至: {save_path}")
        
        # 刷新对比图
        fig, ax = plt.subplots(1, 2, figsize=(10, 5))
        
        ax[0].imshow(target_silhouette[0].cpu().numpy(), cmap='gray')
        ax[0].set_title("Ground Truth Silhouette")
        ax[0].axis("off")
        
        ax[1].imshow(pred_silhouette[0].detach().cpu().numpy(), cmap='gray')
        ax[1].set_title(f"Optimizing... (Epoch {i+1})")
        ax[1].axis("off")
        
        # 保存图片到文件夹，永久留存
        plt.savefig(os.path.join(img_dir, f"silhouette_epoch_{i+1:03d}.png"), dpi=150, bbox_inches="tight")
        plt.show()

print(f"优化完成！所有中间状态的 .obj 文件已保存在 {output_dir} 文件夹中。")
print(f"全部剪影对比图片已保存在 {img_dir} 文件夹中，可直接下载查看。")

In [11]:
import os
import sys
import torch
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import clear_output 

# --- 核心修复：强制添加搜索路径 ---
# 很多时候包装在 site-packages 子目录下，这行代码能自动找到它们
import site
for path in site.getsitepackages():
    if path not in sys.path:
        sys.path.append(path)

def load_obj_manually(filepath):
    """纯 Python 函数读取 obj，返回顶点列表和面列表"""
    verts = []
    faces = []
    with open(filepath, 'r') as f:
        for line in f:
            if line.startswith('v '):
                verts.append([float(x) for x in line.split()[1:4]])
            elif line.startswith('f '):
                # OBJ 的面索引通常是 1-based，转成 0-based
                f_idx = [int(x.split('/')[0]) - 1 for x in line.split()[1:4]]
                faces.append(f_idx)
    return np.array(verts), np.array(faces)

# --- 替代原本加载模型的部分 ---
verts_np, faces_np = load_obj_manually("cow.obj")
print(f"成功读取模型：顶点数 {len(verts_np)}, 面数 {len(faces_np)}")

# 将 NumPy 数据转为 PyTorch 张量（这是唯一需要 torch 的地方）
verts = torch.tensor(verts_np, device=device, dtype=torch.float32)
faces_idx = torch.tensor(faces_np, device=device, dtype=torch.int64)
# 之后你就可以继续使用 verts 和 faces_idx 进行你的优化逻辑了

# 0. 配置设备
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

# 1. 加载目标模型
obj_path = "cow.obj" 
verts, faces, _ = load_obj(obj_path)
faces_idx = faces.verts_idx.to(device)
verts = verts.to(device)
verts = (verts - verts.mean(0)) / max(verts.abs().max(0)[0]) 
cow_mesh = Meshes(verts=[verts], faces=[faces_idx])

# 2. 渲染管线配置
num_views = 20
cameras = FoVPerspectiveCameras(device=device, 
                                R=look_at_view_transform(2.7, torch.zeros(num_views), torch.linspace(-180, 180, num_views))[0], 
                                T=look_at_view_transform(2.7, torch.zeros(num_views), torch.linspace(-180, 180, num_views))[1])

# 调高渲染精度
raster_settings = RasterizationSettings(image_size=256, blur_radius=np.log(1./1e-4 - 1.)*1e-4, faces_per_pixel=50)
rasterizer = MeshRasterizer(cameras=cameras, raster_settings=raster_settings)
shader = SoftSilhouetteShader(blend_params=BlendParams(sigma=1e-4, gamma=1e-4))
target_silhouette = shader(rasterizer(cow_mesh.extend(num_views)), cow_mesh.extend(num_views))[..., 3]

# 3. 高精度初始化：提高网格密度 (ico_sphere 5)
src_mesh = ico_sphere(5, device)
deform_verts = torch.zeros_like(src_mesh.verts_packed(), requires_grad=True)
optimizer = torch.optim.SGD([deform_verts], lr=1.0, momentum=0.9)

# 4. 优化循环
epochs = 500
for i in range(epochs):
    optimizer.zero_grad()
    new_src_mesh = src_mesh.offset_verts(deform_verts)
    pred_silhouette = shader(rasterizer(new_src_mesh.extend(num_views)), new_src_mesh.extend(num_views))[..., 3]
    
    # 【进阶 Loss】：加大几何约束，让模型更“紧致”
    loss_silhouette = ((pred_silhouette - target_silhouette) ** 2).mean()
    loss = loss_silhouette + \
           0.5 * mesh_laplacian_smoothing(new_src_mesh) + \
           0.5 * mesh_edge_loss(new_src_mesh) + \
           0.05 * mesh_normal_consistency(new_src_mesh)
    
    loss.backward()
    optimizer.step()

    # 学习率动态调整
    if i == 250:
        for param_group in optimizer.param_groups:
            param_group['lr'] = 0.5

    if i % 20 == 0 or i == epochs - 1:
        clear_output(wait=True)
        print(f"迭代: {i:03d}/{epochs} | Loss: {loss.item():.4f}")
        fig, ax = plt.subplots(1, 2, figsize=(10, 5))
        ax[0].imshow(target_silhouette[0].cpu().numpy(), cmap='gray')
        ax[0].set_title("Ground Truth")
        ax[1].imshow(pred_silhouette[0].detach().cpu().numpy(), cmap='gray')
        ax[1].set_title(f"Optimizing... (Epoch {i})")
        plt.show()

# 保存最终结果
save_obj("final_cow_model.obj", new_src_mesh.verts_list()[0], new_src_mesh.faces_list()[0])
print("优化完成，已保存为 final_cow_model.obj")

成功读取模型：顶点数 2930, 面数 5856


NameError: name 'load_obj' is not defined

In [7]:
import cv2
import glob
import os

# 1. 设置图片路径和视频输出路径
img_dir = "train_silhouette_imgs"
output_video = "cow_optimization.mp4"

# 2. 获取图片列表 (确保按数字顺序排序)
images = sorted(glob.glob(os.path.join(img_dir, "*.png")))

# 3. 初始化视频写入器
# 使用 cv2.VideoWriter，设置帧率为 15 fps
frame = cv2.imread(images[0])
height, width, layers = frame.shape
video = cv2.VideoWriter(output_video, cv2.VideoWriter_fourcc(*'mp4v'), 15, (width, height))

# 4. 循环写入视频
for img_path in images:
    video.write(cv2.imread(img_path))

video.release()
print(f"动画已生成：{output_video}，请在左侧文件浏览器中下载查看！")

动画已生成：cow_optimization.mp4，请在左侧文件浏览器中下载查看！
